# 03 — VAR Spillover Analysis: Interest Rate Transmission to UK

Estimate a 9-variable VAR on Nelson-Siegel factors (Level, Slope, Curvature)
for US, EU, and UK yield curves to quantify **interest rate spillovers**.

**Key outputs:**
- Impulse Response Functions (IRFs): how UK factors respond to US/EU shocks
- Forecast Error Variance Decomposition (FEVD): share of UK variance from foreign shocks
- Granger causality tests: statistical evidence of spillover direction

**Cholesky ordering (most exogenous first):**
US_Level → US_Slope → US_Curvature → EU_Level → EU_Slope → EU_Curvature → UK_Level → UK_Slope → UK_Curvature

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.api import VAR

from src.var_analysis import (
    VAR_ORDER,
    prepare_var_data,
    adf_tests,
    select_var_lag_detailed,
    estimate_var,
    compute_irfs,
    irf_to_dataframe,
    compute_fevd,
    fevd_summary,
    granger_causality_tests,
    var_diagnostics,
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

## 1. Load and Prepare Data

In [ ]:
# Load NS factors
factors_raw = pd.read_csv('../data/factors/ns_factors.csv', index_col=0, parse_dates=True)
print(f"Raw factors shape: {factors_raw.shape}")

# Separate by region
uk_factors = factors_raw[[c for c in factors_raw.columns if c.startswith('UK_')]].rename(
    columns=lambda x: x.replace('UK_', ''))
us_factors = factors_raw[[c for c in factors_raw.columns if c.startswith('US_')]].rename(
    columns=lambda x: x.replace('US_', ''))
eu_factors = factors_raw[[c for c in factors_raw.columns if c.startswith('EU_')]].rename(
    columns=lambda x: x.replace('EU_', ''))

# Prepare VAR data (weekly frequency)
var_data = prepare_var_data(uk_factors, us_factors, eu_factors, freq='W-FRI')
print(f"VAR data shape: {var_data.shape}")
print(f"Date range: {var_data.index.min()} to {var_data.index.max()}")
print(f"Columns: {list(var_data.columns)}")
var_data.describe()

## 2. Stationarity Tests (ADF)

In [ ]:
# ADF test on levels
adf_levels = adf_tests(var_data)
print("ADF Tests (Levels):")
print(adf_levels[['ADF_stat', 'p_value', 'stationary_5%']].to_string())

print("\n" + "="*60)
print("\nADF Tests (First Differences):")
var_data_diff = var_data.diff().dropna()
adf_diffs = adf_tests(var_data_diff)
print(adf_diffs[['ADF_stat', 'p_value', 'stationary_5%']].to_string())

### Stationarity Decision

Following Diebold, Li & Yue (2008) and the Sims-Stock-Watson (1990) consistency result,
we estimate the VAR in **levels**. This preserves long-run co-movement information.
VAR coefficient estimates remain consistent even with near-unit-root processes.
We use bootstrap confidence bands for IRFs to avoid asymptotic inference issues.

## 3. VAR Lag Selection

In [ ]:
lag_result = select_var_lag_detailed(var_data, max_lags=12)
print("Optimal lag by criterion:")
for criterion, lag in lag_result['recommended'].items():
    print(f"  {criterion}: {lag}")

# Display full summary
lag_result['summary'].summary()

In [ ]:
# Choose lag (use BIC for parsimony, or AIC for fit)
LAGS = lag_result['recommended']['BIC']
print(f"\nSelected lag order: {LAGS} (BIC)")

## 4. VAR Estimation

In [ ]:
var_results = estimate_var(var_data, lags=LAGS)
print(var_results.summary())

## 5. Model Diagnostics

In [ ]:
diag = var_diagnostics(var_results)

print("Durbin-Watson statistics (≈2 means no autocorrelation):")
for var, dw in diag['durbin_watson'].items():
    print(f"  {var}: {dw:.3f}")

print(f"\nPortmanteau test (H0: no residual autocorrelation):")
print(f"  Statistic: {diag['portmanteau']['statistic']:.2f}, p-value: {diag['portmanteau']['p_value']:.4f}")

print(f"\nJarque-Bera normality test:")
print(f"  Statistic: {diag['normality']['statistic']:.2f}, p-value: {diag['normality']['p_value']:.4f}")

## 6. Impulse Response Functions

How do UK factors respond to shocks in US and EU factors?

Orthogonalized (Cholesky) IRFs with 95% bootstrap confidence bands.

In [ ]:
HORIZON = 40  # weeks
irf = compute_irfs(var_results, periods=HORIZON, orth=True)

# Bootstrap confidence intervals (lower, upper bands)
irf_lower, irf_upper = irf.errband_mc(orth=True, repl=500, seed=42)

In [ ]:
# Key IRF plots: Response of UK factors to US and EU shocks
uk_responses = ['UK_Level', 'UK_Slope', 'UK_Curvature']
foreign_shocks = ['US_Level', 'US_Slope', 'US_Curvature',
                  'EU_Level', 'EU_Slope', 'EU_Curvature']

fig, axes = plt.subplots(3, 6, figsize=(20, 10), sharex=True)

var_names = list(var_results.names)

for i, response in enumerate(uk_responses):
    for j, impulse in enumerate(foreign_shocks):
        ax = axes[i, j]
        imp_idx = var_names.index(impulse)
        resp_idx = var_names.index(response)
        
        irf_vals = irf.orth_irfs[:, resp_idx, imp_idx]
        ax.plot(range(HORIZON + 1), irf_vals, 'b-', lw=1.5)
        ax.axhline(0, color='grey', lw=0.5, ls='--')
        ax.fill_between(
            range(HORIZON + 1),
            irf_lower[:, resp_idx, imp_idx],
            irf_upper[:, resp_idx, imp_idx],
            alpha=0.2, color='blue'
        )
        
        if i == 0:
            ax.set_title(f'Shock: {impulse}', fontsize=9)
        if j == 0:
            ax.set_ylabel(f'Response: {response}', fontsize=9)
        if i == 2:
            ax.set_xlabel('Weeks')

plt.suptitle('Orthogonalized IRFs: UK Factor Responses to Foreign Shocks\n(95% Bootstrap CI)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 7. Forecast Error Variance Decomposition

In [ ]:
fevd = compute_fevd(var_results, periods=52)

horizons = [1, 4, 12, 26, 52]

for response in uk_responses:
    print(f"\nFEVD for {response} (% of forecast error variance):")
    summary = fevd_summary(fevd, response, horizons=horizons, var_names=var_names)
    print(summary.round(1).to_string())
    print()

In [ ]:
# FEVD bar charts — aggregate by region
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, response in enumerate(uk_responses):
    ax = axes[idx]
    summary = fevd_summary(fevd, response, horizons=horizons, var_names=var_names)
    
    # Aggregate by region
    us_share = summary[[c for c in summary.columns if c.startswith('US_')]].sum(axis=1)
    eu_share = summary[[c for c in summary.columns if c.startswith('EU_')]].sum(axis=1)
    uk_share = summary[[c for c in summary.columns if c.startswith('UK_')]].sum(axis=1)
    
    agg = pd.DataFrame({'US': us_share, 'EU': eu_share, 'UK (own)': uk_share})
    agg.plot(kind='bar', stacked=True, ax=ax, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
    ax.set_title(f'FEVD: {response}')
    ax.set_xlabel('Horizon (weeks)')
    ax.set_ylabel('% Variance Explained')
    ax.set_ylim(0, 105)
    ax.legend(loc='lower right')
    ax.set_xticklabels(horizons, rotation=0)

plt.suptitle('Forecast Error Variance Decomposition — UK Factors', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Granger Causality Tests

In [ ]:
# Test: Do US factors Granger-cause each UK factor?
us_causing = ['US_Level', 'US_Slope', 'US_Curvature']
eu_causing = ['EU_Level', 'EU_Slope', 'EU_Curvature']

print("Granger Causality: US → UK")
gc_us = granger_causality_tests(var_results, causing=us_causing)
print(gc_us.to_string(index=False))

print("\nGranger Causality: EU → UK")
gc_eu = granger_causality_tests(var_results, causing=eu_causing)
print(gc_eu.to_string(index=False))

print("\nGranger Causality: US+EU → UK")
gc_both = granger_causality_tests(var_results, causing=us_causing + eu_causing)
print(gc_both.to_string(index=False))

## 9. Robustness: Alternative Cholesky Ordering

Re-order as EU → US → UK to check sensitivity.

In [ ]:
# Alternative ordering: EU first
alt_order = [
    'EU_Level', 'EU_Slope', 'EU_Curvature',
    'US_Level', 'US_Slope', 'US_Curvature',
    'UK_Level', 'UK_Slope', 'UK_Curvature',
]
var_data_alt = var_data[alt_order]
var_results_alt = estimate_var(var_data_alt, lags=LAGS)
fevd_alt = compute_fevd(var_results_alt, periods=52)

print("FEVD with alternative ordering (EU → US → UK):")
alt_names = list(var_results_alt.names)
for response in ['UK_Level', 'UK_Slope', 'UK_Curvature']:
    summary = fevd_summary(fevd_alt, response, horizons=[12, 52], var_names=alt_names)
    us_share = summary[[c for c in summary.columns if c.startswith('US_')]].sum(axis=1)
    eu_share = summary[[c for c in summary.columns if c.startswith('EU_')]].sum(axis=1)
    uk_share = summary[[c for c in summary.columns if c.startswith('UK_')]].sum(axis=1)
    print(f"\n  {response} (h=12w): US={us_share.iloc[0]:.1f}%, EU={eu_share.iloc[0]:.1f}%, UK={uk_share.iloc[0]:.1f}%")
    print(f"  {response} (h=52w): US={us_share.iloc[1]:.1f}%, EU={eu_share.iloc[1]:.1f}%, UK={uk_share.iloc[1]:.1f}%")

## 10. Interpretation

### Key Findings

**Level spillovers** (global real rate co-movement):
- The Level factor captures the long-run yield level, heavily influenced by global real rates and inflation expectations.
- US Level shocks are expected to transmit strongly to UK Level, reflecting dollar hegemony and global bond market integration.

**Slope spillovers** (monetary policy transmission):
- The Slope factor proxies the stance of monetary policy (short end vs long end).
- Spillovers here reflect how Fed/ECB rate changes influence BoE expectations and UK market pricing.

**Curvature spillovers** (term premium & risk):
- Curvature captures medium-term humps in the curve, related to uncertainty and term premia.
- These spillovers tend to be weaker and more idiosyncratic.

### US vs EU Influence
- Compare FEVD shares at 52-week horizon to assess relative influence.
- US influence is typically dominant for the Level factor (global risk-free rate).
- EU influence may be relatively stronger for Slope and Curvature (geographic/trade proximity).